In [ ]:
from transition_function_model import (
    setup_transition_function_model,
)
from TD3_Ray import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    # Define paths to environment models ---
    # Trailing commas have been removed to avoid errors.
    ruta_ICE_model = "../src/models_markus/ICE_Model_Update_01"
    ruta_outlook = "../src/models_markus/PG_Model_M1.1_without_EM1_Torque"

    # Create the environment transition function ---
    # Correct variables defined above are used.
    # This function is the "heart" of the environment that the agent will use.
    print("Configuring the environment...")
    t_function = setup_transition_function_model(ruta_ICE_model, ruta_outlook)

    # Load the normalizer and prepare its parameters ---
    # The scaler object containing normalization statistics is loaded.
    print("Loading scaling parameters...")
    scaler = load("../src/escalados/rl.lib")

    # The dictionary of parameters needed by the agent's networks is created.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Initialize Ray (only once per script)
    # Configured to use system memory if necessary (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Allocates 5 GB
    
    bach = 256
    
    U = 64
    B = 32
    
    print("\n--- STARTING FULL TRAINING ---")
    training_start_time = time.time() # [NEW] Starts general timer
    
    # --- Creation and start of the Learner ---
    td3_learner = TD3(
        f_transicio=t_function, 
        version="pls5",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=bach,
        gamma=0.99, 
        tau=0.005, 
        policy_noise=0.2, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Use the 3 CPU cores
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= True
    )
    
    total_training_duration = time.time() - training_start_time # [NEW] Stops the timer

    print("\n--- TOTAL TRAINING FINISHED ---")
    print(f"✅ The complete training process took: {total_training_duration:.2f} seconds ({total_training_duration / 60:.2f} minutes).") # [NEW] Prints the result
    
    # Start asynchronous training
    td3_learner.learn(total_timesteps=1000000, learning_starts=bach*3, train_freq=int(bach*U)*2, gradient_steps=5000)

    # Stop Ray when finished
    ray.shutdown()

## EVALUATION

In [ ]:
from transition_function_model import (
    setup_transition_function_model,
)
from TD3_Ray_EVALUATION import *
from joblib import load

import time

import ray


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # You only need the last one
    os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
    
    
    # Define paths to environment models ---
    # Trailing commas have been removed to avoid errors.
    ruta_ICE_model = "../src/models_markus/ICE_Model_Update_01"
    ruta_outlook = "../src/models_markus/PG_Model_M1.1_without_EM1_Torque"

    # Create the environment transition function ---
    # Correct variables defined above are used.
    # This function is the "heart" of the environment that the agent will use.
    print("Configuring the environment...")
    t_function = setup_transition_function_model(ruta_ICE_model, ruta_outlook)

    # Load the normalizer and prepare its parameters ---
    # The scaler object containing normalization statistics is loaded.
    print("Loading scaling parameters...")
    scaler = load("../src/escalados/rl.lib")

    # The dictionary of parameters needed by the agent's networks is created.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    
    # --- Base Variables ---
    BATCH_SIZE = 256

    
    # --- STEP 1: Configure environment for profiling BEFORE starting Ray ---
    PROFILE_TRAINING = True # Define if you want to do profiling here

    # --- STEP 2: Start Ray ONLY ONCE ---
    print("Starting Ray...")
    if ray.is_initialized():
        ray.shutdown()

    ray.init(object_store_memory=5 * 10**9, include_dashboard=False)
    
    


    # --- STEP 3: Now that Ray is active, create the agent and its workers ---
    print("Configuring TD3 agent...")
    td3_learner = TD3(
        f_transicio=t_function, 
        version="pls5",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=256,
        gamma=0.99, 
        tau=0.005, 
        policy_noise=0.2, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3,
        U=64,
        B=32,
        early_stop=500,
        reuse_warmup_buffer= True,
        profile_training=PROFILE_TRAINING 
    )

    print("\n--- STARTING FULL TRAINING ---")
    training_start_time = time.time() # [NEW] Starts general timer
    
    
    # --- STEP 4: Start training ---
    td3_learner.learn(total_timesteps=10000, learning_starts=BATCH_SIZE*10, train_freq=BATCH_SIZE, gradient_steps=BATCH_SIZE)

    total_training_duration = time.time() - training_start_time # [NEW] Stops the timer

    
    print("\n--- TOTAL TRAINING FINISHED ---")
    print(f"✅ The complete training process took: {total_training_duration:.2f} seconds ({total_training_duration / 60:.2f} minutes).") # [NEW] Prints the result
    
    # Start asynchronous training
    # --- STEP 5: Stop Ray when finished ---
    ray.shutdown()